# 🎬 ПоискКино API — Jupyter Notebook

Готовый пример работы с API ПоискКино. Токен вставляется в первую ячейку.

**Источник документации:** https://api.poiskkino.dev/documentation


In [ ]:
# Установим необходимые библиотеки
%pip install -q requests pandas matplotlib


In [ ]:
# ВСТАВЬТЕ СЮДА СВОЙ API-ТOКЕН
API_KEY = "ВСТАВЬТЕ_ТОКЕН_СЮДА"

BASE_URL = "https://api.poiskkino.dev"
HEADERS = {"X-API-KEY": API_KEY}

if API_KEY == "ВСТАВЬТЕ_ТОКЕН_СЮДА":
    print("⚠️ Вставьте API-токен в переменную API_KEY")
else:
    print("✅ Токен задан")


In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

def api_get(endpoint, params=None):
    """GET-запрос к API ПоискКино с обработкой основных ошибок."""
    response = requests.get(
        BASE_URL + endpoint,
        headers=HEADERS,
        params=params,
        timeout=30
    )
    if response.status_code == 401:
        raise Exception("401: неверный или отсутствующий API-токен")
    if response.status_code == 403:
        raise Exception("403: доступ запрещён или превышен лимит")
    response.raise_for_status()
    return response.json()


## 1. Поиск фильма по названию


In [ ]:
query = input("Введите название фильма: ").strip()
data = api_get("/v1.4/movie/search", {"query": query, "page": 1, "limit": 10})
data


In [ ]:
# Удобная таблица результатов
docs = data.get("docs", [])
rows = []
for m in docs:
    rows.append({
        "ID": m.get("id"),
        "Название": m.get("name"),
        "Год": m.get("year"),
        "Рейтинг КП": (m.get("rating") or {}).get("kp"),
        "Рейтинг IMDb": (m.get("rating") or {}).get("imdb"),
        "Тип": m.get("type")
    })
pd.DataFrame(rows)


## 2. Получение фильма по ID


In [ ]:
movie_id = input("Введите ID фильма (например, 692861): ").strip()
movie = api_get(f"/v1.4/movie/{movie_id}")
movie


## 3. Поиск с фильтрами


In [ ]:
year = input("Год фильма (например, 2024): ").strip()
min_rating = input("Минимальный рейтинг КП (например, 7): ").strip()

params = {"page": 1, "limit": 20}
if year:
    params["year"] = year
if min_rating:
    params["rating.kp"] = f"{min_rating}-10"

filtered = api_get("/v1.4/movie", params)
filtered


In [ ]:
docs = filtered.get("docs", [])
rows = []
for m in docs:
    rating = m.get("rating") or {}
    rows.append({
        "ID": m.get("id"),
        "Название": m.get("name"),
        "Год": m.get("year"),
        "КП": rating.get("kp"),
        "IMDb": rating.get("imdb"),
        "Жанры": ", ".join(g.get("name", "") for g in (m.get("genres") or []))
    })
df = pd.DataFrame(rows)
df


## 4. Случайный фильм


In [ ]:
random_movie = api_get("/v1.4/movie/random")
print("Название:", random_movie.get("name"))
print("Год:", random_movie.get("year"))
print("ID:", random_movie.get("id"))
print("Рейтинг КП:", (random_movie.get("rating") or {}).get("kp"))
print("Рейтинг IMDb:", (random_movie.get("rating") or {}).get("imdb"))
print("Описание:", random_movie.get("description"))


## 5. Поиск актёра / режиссёра


In [ ]:
person_name = input("Введите имя актёра или режиссёра: ").strip()
persons = api_get("/v1.4/person/search", {"query": person_name, "page": 1, "limit": 10})
persons


In [ ]:
rows = []
for p in persons.get("docs", []):
    rows.append({
        "ID": p.get("id"),
        "Имя": p.get("name"),
        "Имя на английском": p.get("enName"),
        "Дата рождения": p.get("birthday"),
        "Профессии": ", ".join(x.get("name", "") for x in (p.get("profession") or []))
    })
pd.DataFrame(rows)


## 6. Простая визуализация рейтингов найденных фильмов


In [ ]:
if not df.empty and "КП" in df.columns:
    plot_df = df.dropna(subset=["КП"]).head(10)
    if not plot_df.empty:
        plt.figure(figsize=(10, 5))
        plt.bar(plot_df["Название"].astype(str), plot_df["КП"].astype(float))
        plt.xticks(rotation=45, ha="right")
        plt.ylabel("Рейтинг Кинопоиска")
        plt.title("Рейтинги найденных фильмов")
        plt.tight_layout()
        plt.show()


## Готово

Для работы замените `ВСТАВЬТЕ_ТОКЕН_СЮДА` на ваш токен и выполняйте ячейки сверху вниз.

⚠️ Не публикуйте токен в GitHub или открытом Notebook.
